### Recommender Systems

Architecture for ranking/recommendation at scale: the two-stage candidate-generation-then-ranking funnel, two-tower retrieval models, cold-start handling, in-batch/hard-negative mining, and multi-objective fusion at serving. The bi-encoder-vs-cross-encoder tradeoff in `rag-pipeline.ipynb` is the same *shape* of problem (cheap-and-approximate at scale vs. expensive-and-precise on a shortlist) in a different domain.

## Two-stage funnel: candidate generation vs. ranking

#### 0. Core idea

You can't score every item against every user with a full cross-feature model at serving time, the latency and compute don't scale (10M items x every request). The standard fix is a two-stage funnel:

```
10,000,000 items
      |  stage 1: candidate generation (cheap, approximate, e.g. two-tower + ANN search)
  ~500 candidates
      |  stage 2: ranking (expensive, precise, full cross-feature model, only run on the shortlist)
   ~10 shown to the user
```
Stage 1 optimizes for recall (don't miss anything good) over a huge catalog cheaply; stage 2 optimizes for precision (get the order right) over a small shortlist expensively. Getting this split wrong in either direction (an inaccurate stage 1 that drops good candidates, or a stage 2 too slow to run even on 500 items) breaks the whole pipeline.

## Two-tower model vs. DCN / wide-and-deep

#### 0. Core idea

**Two-tower**: a separate neural network encodes the user into a vector and another encodes the item into a vector, of the SAME dimension, and the score is just their dot product. Because item vectors don't depend on the specific user, every item embedding can be pre-computed offline and indexed for approximate nearest-neighbor (ANN) search, this is what makes two-tower usable as the cheap stage-1 candidate generator.

**DCN / wide-and-deep**: a single network takes user AND item features together and lets them cross (explicit feature crosses, or deep interaction layers). More accurate, because the model can learn interactions between specific user and item features, but the score can't be pre-computed since it depends on both sides jointly, every candidate has to be scored individually at request time. That's why this is the stage-2 ranker, run only on the stage-1 shortlist, never the whole catalog.

Toy two-tower scoring, user vector `u = [1.0, 0.5]`, three candidate item vectors already in the same embedding space:

In [ ]:
import numpy as np

u = np.array([1.0, 0.5])
items = {
    "item_relevant":   np.array([1.0, 0.5]),
    "item_somewhat":   np.array([0.2, 0.1]),
    "item_irrelevant": np.array([-1.0, 0.5]),
}
for name, v in items.items():
    print(name, "dot-product score:", np.dot(u, v))

The relevant item scores highest by construction (its vector is nearly identical to the user vector), the irrelevant one scores negative. This dot-product-only structure is exactly what ANN indexes (built for cosine/dot-product similarity) can search over in sub-linear time, no such shortcut exists for a DCN-style score that mixes user and item features together.

## In-batch negatives and hard-negative mining

#### 0. Core idea

Training a two-tower model needs negative (non-matching) user-item pairs, not just the positives observed in interaction logs. **In-batch negatives**: within a training batch, treat every OTHER item in the batch as a negative for each user, cheap (no extra sampling) but the negatives are just whatever happened to be in the batch, usually easy to distinguish from the true positive.

Toy: 4 users, 4 items, the diagonal is the true (user, item) positive pair; every off-diagonal cell is an in-batch negative.

In [ ]:
import numpy as np

U = np.array([
    [1.0, 0.2],
    [0.1, 1.0],
    [0.8, 0.8],
    [-0.5, 0.5],
])
I = np.array([
    [1.0, 0.1],   # true match for user 0
    [0.0, 0.9],   # true match for user 1
    [0.9, 0.7],   # true match for user 2
    [-0.6, 0.4],  # true match for user 3
])
scores = U @ I.T  # row = user, col = item; off-diagonal = in-batch negatives
print("score matrix (rows=users, cols=items):")
print(np.round(scores, 3))

def softmax(x):
    e = np.exp(x - x.max())
    return e / e.sum()

for row in range(4):
    probs = softmax(scores[row])
    loss = -np.log(probs[row])
    print(f"user {row}: softmax over in-batch items = {np.round(probs, 3)}, loss on true positive = {round(loss, 3)}")

#### 1. Why hard negatives matter

Look at user 0: item 2 (score 1.04) actually outscores item 0, the true positive (score 1.02), even though item 2 belongs to a different user. Item 2 is a **hard negative** for user 0, similar enough in embedding space to be genuinely confusable, and it's exactly this kind of near-miss that in-batch sampling might not surface reliably (whether it appears depends on random batch composition). Hard-negative mining deliberately searches for and injects negatives like this (e.g. the top-scoring non-matches from a previous training round) into every batch, forcing the model to sharpen the decision boundary in the region that actually matters at serving time, near the top of the ranked list, rather than getting an artificially easy loss from batches full of obviously-irrelevant negatives.

## Cold-start

#### 0. Core idea

A brand-new item (or user) has no interaction history, so an ID-based embedding for it is literally untrained, close to its random initialization, and useless for scoring. The standard fallback is a CONTENT-based proxy: derive a proxy embedding from item metadata (category, text, image) instead of interaction history, often by averaging the learned embeddings of similar warm items in the same category.

In [ ]:
import numpy as np

# warm items in the same category, with real interaction-learned embeddings
warm_items_in_category = np.array([
    [0.85, 0.55],
    [0.95, 0.65],
    [0.90, 0.60],
])
category_centroid = warm_items_in_category.mean(axis=0)
print("category centroid, used as the cold item's proxy embedding:", category_centroid)

untrained_id_embedding = np.array([0.02, -0.01])  # what the item's own ID-embedding looks like pre-training
print("vs. the item's own untrained ID-embedding:            ", untrained_id_embedding)

The category-centroid proxy lands close to where the item's OWN embedding will eventually converge once it accumulates real interactions, giving it a reasonable score from day one, while the untrained ID-embedding carries no signal at all and would rank the item essentially at random.

## Multi-objective fusion at serving

#### 0. Core idea

Optimizing the ranker purely for click probability tends to surface clickbait and under-values items with lower click rate but higher downstream value (revenue, long-term engagement, retention). Fusion combines several predicted objectives into one serving-time score, typically a weighted sum: `score = w_click * p_click + w_engage * p_engage + w_value * business_value`, with weights set by the business, not learned end-to-end.

Toy: three candidates ranked by CTR alone vs. by a fused score with `w_click=1.0, w_engage=1.0, w_value=0.5`:

In [ ]:
candidates = {
    "A": dict(p_click=0.30, p_engage=0.10, value=1.0),
    "B": dict(p_click=0.20, p_engage=0.05, value=5.0),
    "C": dict(p_click=0.15, p_engage=0.20, value=0.5),
}
print("CTR-only ranking:", sorted(candidates, key=lambda k: -candidates[k]["p_click"]))

w_click, w_engage, w_value = 1.0, 1.0, 0.5
fused = {k: w_click*v["p_click"] + w_engage*v["p_engage"] + w_value*v["value"] for k, v in candidates.items()}
print("fused scores:", fused)
print("fused ranking:  ", sorted(fused, key=lambda k: -fused[k]))

#### 1. The tradeoff

Item B has the lowest click probability of the three but the highest business value, and the fused score correctly promotes it to #1, the outcome you actually want. The cost: the weights (`w_click`, `w_engage`, `w_value`) are a real business decision with no single "correct" answer, get them wrong and you either starve high-value-but-lower-CTR items (weights too click-heavy) or flood users with high-value junk they never engage with (weights too value-heavy). In practice these weights get tuned against a holdout business metric (`ab-testing.ipynb`), not set once and left alone.